# 🔬 Dissecting a Tumor's Ecosystem
### From single cells to 1,000 patients

A tumor isn't just cancer cells — it's a whole **neighborhood**: cancer cells, immune cells that fight them, and support cells like fibroblasts and blood vessels.

Today you'll use **real single-cell data** from breast tumors to figure out *who lives in* 1,000 patient tumors — and discover how two kinds of breast cancer are built differently.

*Run each grey cell with ▶ (or Shift+Enter). Read the text as you go.*

## Part 0 · Setup
Run this once. It loads our tools and downloads the data we prepared for you.

In [ ]:
%matplotlib inline
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy.optimize import nnls
from scipy.stats import mannwhitneyu
sns.set_style('white')
RAW = 'https://raw.githubusercontent.com/bts76-pitt/bioinfo-bootcamp-2026-data/main/'
print('Tools loaded ✅')

This little function is the heart of the lesson — it figures out the **recipe** of a tumor. We'll see how it works in Part 3. For now just run it.

In [ ]:
def deconvolve(sig, bulk):
    '''Estimate what fraction of each cell type makes up each tumor.'''
    shared = sorted(set(sig.index) & set(bulk.index))
    sig, bulk = sig.loc[shared], bulk.loc[shared]
    scale = sig.max(axis=1).replace(0, 1)        # put every gene on a 0-1 scale
    S = sig.div(scale, axis=0).values
    out = {}
    for sample in bulk.columns:
        b = (bulk[sample] / scale).values
        coef, _ = nnls(S, b)                     # solve bulk = signature x fractions
        out[sample] = coef / coef.sum() if coef.sum() > 0 else coef
    return pd.DataFrame(out, index=sig.columns).T
print('deconvolve() ready')

In [ ]:
sig  = pd.read_csv(RAW + 'signature_matrix.csv', index_col=0)
viz  = pd.read_csv(RAW + 'atlas_umap.csv.gz')
bulk = pd.read_csv(RAW + 'tcga_brca_bulk.csv', index_col=0)
clin = pd.read_csv(RAW + 'tcga_brca_clinical.csv')
print('Single cells for the map :', viz.shape[0], 'cells')
print('Patient tumors (bulk)    :', bulk.shape[1], 'patients')
print('Cell types in our cookbook:', list(sig.columns))

## Part 1 · Meet the cells 🧫
Scientists can now read the genes of **one cell at a time**. Each dot below is a single real cell from a breast tumor, placed so that similar cells sit close together (this is called a **UMAP**). The colors are the cell types experts identified.

In [ ]:
palette = dict(zip(sorted(viz.cell_type.unique()),
                   sns.color_palette('tab10', viz.cell_type.nunique())))
fig, ax = plt.subplots(figsize=(8, 6.5))
for ct, d in viz.groupby('cell_type'):
    ax.scatter(d.UMAP1, d.UMAP2, s=4, color=palette[ct], label=ct, alpha=0.6)
ax.legend(markerscale=4, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
ax.set(title='Every dot is one cell from a breast tumor', xticks=[], yticks=[])
sns.despine(left=True, bottom=True); plt.tight_layout(); plt.show()

**How did experts know which cell is which?** Each cell type switches on signature genes.
Let's color the *same map* by one gene at a time — bright = that gene is ON.

- **EPCAM** marks epithelial (cancer) cells
- **CD3D** marks T cells (immune)
- **COL1A1** marks fibroblasts (support cells that make collagen)

Watch how each gene lights up a *different* island — that's how we read cell identity.

In [ ]:
def color_by_gene(gene):
    fig, ax = plt.subplots(figsize=(6, 5))
    s = ax.scatter(viz.UMAP1, viz.UMAP2, s=4, c=viz[gene], cmap='viridis')
    plt.colorbar(s, label=f'{gene} level')
    ax.set(title=f'{gene}', xticks=[], yticks=[]); sns.despine(left=True, bottom=True)
    plt.tight_layout(); plt.show()

for g in ['EPCAM', 'CD3D', 'COL1A1']:
    color_by_gene(g)

👉 **Your turn:** change the gene below to **CDH1** (a famous breast-cancer gene) and run it. Which cells light up? Keep that in mind — we'll come back to CDH1 at the end.

In [ ]:
color_by_gene('CDH1')

## Part 2 · The problem with patient data 🥤
Single-cell data is amazing, but we only have it for a few tumors. For **thousands** of patients we instead have **bulk** data: the whole tumor blended into one smoothie, so every cell's genes got mixed together. Here it is — genes down the side, patients across the top. **There are no cell types anywhere.** The recipe is lost.

In [ ]:
print('Bulk tumors:', bulk.shape[0], 'genes x', bulk.shape[1], 'patients')
bulk.iloc[:5, :4]

To get the recipe back we need a **cookbook**: the typical gene 'fingerprint' of each cell type, learned from the single cells. Each column below is one cell type; bright rows are the genes it makes a lot of. See the blocks? Each cell type has its own signature.

In [ ]:
scaled = sig.div(sig.max(axis=1).replace(0, 1), axis=0)  # put each gene on a 0-1 scale
order = np.argsort(sig.values.argmax(axis=1))           # group genes by the type they mark
fig, ax = plt.subplots(figsize=(5.5, 7))
sns.heatmap(scaled.iloc[order], cmap='magma', yticklabels=False,
            cbar_kws={'label': 'relative level (0-1)'}, ax=ax)
ax.set(title='The cookbook: each cell types fingerprint', xlabel='', ylabel='genes')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## Part 3 · Deconvolution: getting the recipe back 🧮
Here's the big idea, as one equation:

$$\text{bulk smoothie} \;\approx\; \text{cookbook} \times \text{recipe}$$

We **know** the smoothie (bulk) and the cookbook (signature). We **solve** for the recipe — the fraction of each cell type. It's like tasting a smoothie and working out how much of each fruit went in. Let's do one patient first.

In [ ]:
one = bulk.columns[0]
recipe = deconvolve(sig, bulk[[one]])
ax = recipe.T.plot.bar(legend=False, color='teal', figsize=(7, 4))
ax.set(title=f'Estimated cell-type recipe for patient {one}', ylabel='fraction of tumor')
plt.xticks(rotation=45, ha='right'); sns.despine(); plt.tight_layout(); plt.show()
print('Fractions add up to:', round(float(recipe.iloc[0].sum()), 2))

Now let's run it on **all 1,000+ patients at once**. This is the real analysis — *you're* running it.

In [ ]:
fractions = deconvolve(sig, bulk)
print('Cell-type recipes for', fractions.shape[0], 'patients:')
fractions.round(3).head()

## Part 4 · Two kinds of breast cancer 🎗️
Breast tumors come in types. The two most common:
- **IDC** (ductal) — the most common kind
- **ILC** (lobular) — grows in single-file lines, ~10% of cases

Do their ecosystems differ? Let's attach each patient's type to their recipe and compare.

In [ ]:
frac = fractions.copy()
frac['type'] = clin.set_index('sample')['subtype'].reindex(frac.index).values
frac = frac.dropna(subset=['type'])
cell_types = [c for c in fractions.columns]
comp = frac.groupby('type')[cell_types].mean()
comp.plot.bar(stacked=True, colormap='tab10', figsize=(6, 5))
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
plt.title('Average tumor make-up'); plt.ylabel('fraction'); plt.xticks(rotation=0)
sns.despine(); plt.tight_layout(); plt.show()

The pies look similar! Let's test each cell type for a real difference (a small p-value means the difference is unlikely to be luck).

In [ ]:
for ct in cell_types:
    a = frac[frac.type=='ILC'][ct]; b = frac[frac.type=='IDC'][ct]
    p = mannwhitneyu(a, b).pvalue
    flag = '  <-- different!' if p < 0.01 else ''
    print(f'{ct:18} IDC={b.mean():.3f}  ILC={a.mean():.3f}  p={p:.0e}{flag}')

So the **ecosystems are mostly similar** — a few support cells differ a little. If the cell mix is so similar, what actually makes lobular cancer different? Let's look at the **cancer genes themselves** in the bulk data. Watch **CDH1** — the gene that *defines* lobular cancer.

In [ ]:
CANCER_GENES = ['CDH1', 'ESR1', 'ERBB2', 'MKI67', 'TP53', 'PIK3CA']
sub = clin.set_index('sample')['subtype']
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, g in zip(axes.ravel(), CANCER_GENES):
    if g not in bulk.index:
        ax.set_visible(False); continue
    df = pd.DataFrame({'expr': np.log1p(bulk.loc[g]), 'type': sub.reindex(bulk.columns).values}).dropna()
    p = mannwhitneyu(df[df.type=='ILC'].expr, df[df.type=='IDC'].expr).pvalue
    sns.boxplot(data=df, x='type', y='expr', order=['IDC','ILC'], hue='type',
                legend=False, ax=ax, palette={'IDC':'#bbbbbb','ILC':'#d1495b'})
    ax.set(title=f'{g}  (p={p:.0e})', xlabel='', ylabel='expression (log)')
sns.despine(); plt.tight_layout(); plt.show()

**CDH1 collapses in ILC** — by far the biggest difference. CDH1 makes E-cadherin, the glue that holds epithelial cells together. Lose the glue and cells stop sticking — that's why lobular cancer grows in single-file lines. 🧬

## Part 5 · Real difference, or just a different mix? 🕵️
Here's the trap with bulk data. When a gene looks different between ILC and IDC, there are **two possible explanations**:

1. the cells *themselves* changed it (**real**, cell-intrinsic), or
2. the tumor just has a *different mix* of cells (a **mirage**).

Deconvolution lets us tell them apart. Let's put CDH1 on trial.

**Could low CDH1 in ILC just be a mirage** — maybe ILC tumors simply have fewer cancer cells (the cells that make CDH1)? Let's check the cancer-cell fraction in each type.

In [ ]:
for t in ['IDC', 'ILC']:
    idx = frac[frac.type==t].index
    print(f'{t}:  cancer-cell fraction = {frac.loc[idx, "Cancer Epithelial"].mean():.3f}'
          f'   |   CDH1 level = {bulk.loc["CDH1", idx].mean():.1f}')

**The verdict:** the cancer-cell fraction is *basically identical* in ILC and IDC — yet CDH1 is several times lower in ILC. Same amount of cancer cells, far less CDH1 → the cancer cells **genuinely switched it off**. That's *real* biology, the hallmark of lobular cancer. ✅

**Now a gene that IS a mirage.** Blood-vessel genes like **PECAM1** look different between tumors — but only because tumors have different amounts of blood vessels. Watch PECAM1 track the endothelial (blood-vessel) fraction: when there's more vessel, there's more PECAM1. No cell changed — the *mix* did.

In [ ]:
common = [s for s in frac.index if s in bulk.columns]
r = np.corrcoef(frac.loc[common, 'Endothelial'], bulk.loc['PECAM1', common])[0, 1]
plt.figure(figsize=(6, 5))
plt.scatter(frac.loc[common, 'Endothelial'], bulk.loc['PECAM1', common], s=10, alpha=0.5, color='#2e8b57')
plt.xlabel('blood-vessel fraction (from deconvolution)'); plt.ylabel('PECAM1 level (bulk)')
plt.title(f'PECAM1 just tracks the blood-vessel fraction  (r = {r:.2f})')
sns.despine(); plt.tight_layout(); plt.show()

## 🎉 What you did today
- Saw that a tumor is an **ecosystem** of many cell types
- Used a single-cell **cookbook** to **deconvolve** 1,000+ bulk tumors — recovering the recipe from the smoothie
- Found that ILC and IDC tumors have **similar ecosystems**
- Showed **CDH1 loss in lobular cancer is real**, not a mixing artifact — the defining event of the disease
- Learned the detective skill that **composition can fool bulk data**, and how to catch it

*This is exactly how real researchers mine thousands of patient tumors. Deconvolution gives estimates, not exact counts — but it turns a blended smoothie back into a list of ingredients.* 🔬